In [1]:
# cell 1 - imports and the two paths we need
import os, random, shutil                           # basic python tools
import numpy as np                                  # arrays
import pandas as pd                                 # tables
import cv2                                          # image reading
from tqdm import tqdm                               # progress bars

IMG_ROOT = "/kaggle/input/datasets/deepapaneru/diabetic-retinopathy-datasets"   # aptos images
MASTER = "/kaggle/input/datasets/bikramchapagain/master-e1/aptos_e1_master.csv"           # master table

print(os.listdir(IMG_ROOT))                         # should show the train/val/test image folders

['APTOS2019', 'messidor-1', 'multimodal']


In [2]:
# cell 2 - the master csv has the old paths, so find every image again by its name
m_all = pd.read_csv(MASTER)                         # read the master table
print("old path example:", m_all.path[0])           # path from the first notebook

found = {}                                          # image name (no extension) -> full path
for dp, _, files in os.walk(IMG_ROOT):              # walk through the aptos folder only
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            found[os.path.splitext(f)[0]] = os.path.join(dp, f)
print("images found:", len(found))                  # should be 3662

m_all["path"] = m_all["id_code"].astype(str).map(found)   # put the new path next to each id
print("still missing:", m_all.path.isna().sum())    # must be 0
print(m_all.split.value_counts().to_dict())         # {'train': 2930, 'val': 366, 'test': 366}

old path example: /kaggle/input/datasets/deepapaneru/dr-datasets/archive/train_images/train_images/1ae8c165fd53.png
images found: 4117
still missing: 0
{'train': 2930, 'val': 366, 'test': 366}


In [3]:
# cell 3 - all settings in one place (keep these the same as the seed 42 run)
import torch                                        # main deep learning library
import torch.nn as nn                               # layers and loss functions
from torch.utils.data import Dataset, DataLoader    # to feed images in batches
import torchvision                                  # pretrained efficientnet is here
from torchvision import transforms as T             # augmentation tools
from sklearn.metrics import cohen_kappa_score       # used for QWK

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # use gpu if we have one
IMG = 512                                           # image size
BATCH = 16                                          # batch size (use 8 if memory runs out)
EPOCHS = 25                                         # max epochs
PATIENCE = 5                                        # stop if val QWK does not improve for 5 epochs
LR = 3e-4                                           # learning rate
WD = 1e-4                                           # weight decay
PREP = "both"                                       # ben graham + clahe
SEED = 2024                                         # this notebook's seed

def set_seed(s):                                    # makes the run repeatable
    random.seed(s)                                  # python random
    np.random.seed(s)                               # numpy random
    torch.manual_seed(s)                            # torch random
    torch.cuda.manual_seed_all(s)                   # gpu random

set_seed(SEED)                                      # apply the seed now

m_train = m_all[m_all.split == "train"].reset_index(drop=True)   # training rows
m_val = m_all[m_all.split == "val"].reset_index(drop=True)       # validation rows
m_test = m_all[m_all.split == "test"].reset_index(drop=True)     # test rows
print(DEVICE, len(m_train), len(m_val), len(m_test))             # 2930 366 366



cuda 2930 366 366


In [4]:
# cell 4 - preprocessing, same for train, val and test
def crop_black(img):                                # removes the black border around the retina
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)    # gray copy to find the retina
    ys, xs = np.where(gray > 10)                    # pixels that are not black
    if len(xs) == 0:                                # if the image is fully black
        return img                                  # leave it as it is
    return img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # cut to the bounding box

def make_square(img):                               # pads the short side so the retina is not stretched
    h, w = img.shape[:2]                            # current height and width
    d = abs(h - w)                                  # how much padding we need
    a, b = d // 2, d - d // 2                       # split the padding on both sides
    if h > w:                                       # taller than wide
        return cv2.copyMakeBorder(img, 0, 0, a, b, cv2.BORDER_CONSTANT, value=0)   # pad left and right
    return cv2.copyMakeBorder(img, a, b, 0, 0, cv2.BORDER_CONSTANT, value=0)       # pad top and bottom

def preprocess(path):                               # full pipeline for one image
    img = cv2.imread(path)                          # read the file (BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)      # convert to RGB
    img = make_square(crop_black(img))              # crop border then make it square
    img = cv2.resize(img, (IMG, IMG))               # resize to 512 x 512
    if PREP in ("ben", "both"):                     # Ben Graham step
        sigma = IMG / 2 / 30                        # sigma = radius / 30
        blur = cv2.GaussianBlur(img, (0, 0), sigma) # blurred copy
        img = cv2.addWeighted(img, 4, blur, -4, 128)   # 4*I - 4*blur + 128
    if PREP in ("clahe", "both"):                   # CLAHE step
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)  # work on lightness only
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))   # settings from the methodology
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])    # boost local contrast on L channel
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)  # back to RGB
    return img                                      # uint8 image, 512 x 512 x 3

test_img = preprocess(m_train.path[0])              # try one image
print(test_img.shape, test_img.dtype)               # (512, 512, 3) uint8

(512, 512, 3) uint8


In [5]:
# cell 5 - augmentation (train only), dataset class and loaders
train_aug = T.Compose([                             # random changes, only for training images
    T.RandomHorizontalFlip(),                       # random left-right flip
    T.RandomVerticalFlip(),                         # random up-down flip
    T.RandomRotation(30),                           # rotate up to 30 degrees
    T.ColorJitter(brightness=0.2, contrast=0.2),    # small brightness and contrast change
])                                                  # no blur, as decided in the literature table

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)   # ImageNet mean
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)    # ImageNet std

class DRData(Dataset):                              # gives one image and its labels at a time
    def __init__(self, table, train):               # table = dataframe, train = use augmentation or not
        self.paths = table.path.tolist()            # list of image paths
        self.labels = table.label.astype(int).tolist()   # list of grades 0-4
        self.train = train                          # remember the mode

    def __len__(self):                              # number of images
        return len(self.paths)

    def __getitem__(self, i):                       # load image number i
        img = preprocess(self.paths[i])             # run the preprocessing
        x = torch.from_numpy(img).permute(2, 0, 1)  # to channel-first tensor (uint8)
        if self.train:                              # augmentation only while training
            x = train_aug(x)
        x = x.float() / 255.0                       # scale to 0-1
        x = (x - MEAN) / STD                        # normalise like ImageNet
        y = self.labels[i]                          # the grade
        levels = torch.tensor([float(y > k) for k in range(4)])   # ordinal targets: grade > 0, > 1, > 2, > 3
        return x, levels, y                         # image, ordinal targets, plain grade

train_loader = DataLoader(DRData(m_train, True), batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)   # shuffled for training
val_loader = DataLoader(DRData(m_val, False), batch_size=BATCH, shuffle=False,
                        num_workers=2, pin_memory=True)                     # fixed order for val
test_loader = DataLoader(DRData(m_test, False), batch_size=BATCH, shuffle=False,
                         num_workers=2, pin_memory=True)                    # fixed order for test

xb, lb, yb = next(iter(train_loader))               # pull one batch to check shapes
print(xb.shape, lb.shape, yb[:8])                   # (16,3,512,512) (16,4)

torch.Size([16, 3, 512, 512]) torch.Size([16, 4]) tensor([2, 0, 0, 2, 0, 2, 0, 4])


In [6]:
# cell 6 - EfficientNet-B0 with a CORAL ordinal head
class CoralEffNet(nn.Module):
    def __init__(self):
        super().__init__()                          # start the parent class
        base = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")   # pretrained (needs internet ON)
        self.features = base.features               # keep only the convolution part
        self.pool = nn.AdaptiveAvgPool2d(1)         # global average pooling
        self.drop = nn.Dropout(0.3)                 # dropout against overfitting
        self.fc = nn.Linear(1280, 1, bias=False)    # one shared score z = w^T h
        self.bias = nn.Parameter(torch.linspace(1.5, -1.5, 4))   # 4 thresholds, start in decreasing order

    def forward(self, x):
        h = self.pool(self.features(x)).flatten(1)  # 1280-d feature per image
        z = self.fc(self.drop(h))                   # shared score, shape (batch, 1)
        return z + self.bias                        # logits for the 4 thresholds, shape (batch, 4)

counts = m_train.label.astype(int).value_counts().sort_index()      # images per grade in train
CLASS_W = torch.tensor((len(m_train) / (5 * counts)).values,
                       dtype=torch.float32).to(DEVICE)              # omega_c = N / (K * n_c)
print(CLASS_W)                                      # expect 0.4086, 1.9533, 0.7252, 3.8052, 2.5043

def coral_loss(logits, levels, grades):             # weighted CORAL loss
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, levels, reduction="none").sum(1)    # binary loss summed over the 4 thresholds
    w = CLASS_W[grades]                             # weight of each sample by its true grade
    return (bce * w).mean()                         # average over the batch

tensor([0.4086, 1.9533, 0.7252, 3.8052, 2.5043], device='cuda:0')


In [7]:
# cell 7 - helper functions for prediction and training
def predict(model, loader):                         # returns threshold probabilities and true grades
    model.eval()                                    # evaluation mode (no dropout)
    all_p, all_y = [], []                           # collectors
    with torch.no_grad():                           # no gradients needed
        for x, _, y in loader:
            x = x.to(DEVICE)                        # move images to gpu
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):   # mixed precision
                out = model(x)                      # logits
            all_p.append(torch.sigmoid(out.float()).cpu())   # threshold probabilities
            all_y.append(y)                         # true grades
    return torch.cat(all_p).numpy(), torch.cat(all_y).numpy()   # as numpy arrays

def qwk(p, y):                                      # quadratic weighted kappa from threshold probs
    pred = (p > 0.5).sum(1)                         # predicted grades
    return cohen_kappa_score(y, pred, weights="quadratic")   # QWK

def train_model(seed, out_dir):                     # full training run for one seed
    set_seed(seed)                                  # repeatable run
    model = CoralEffNet().to(DEVICE)                # new model
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)   # optimiser
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))  # LR goes up then down
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")       # for mixed precision
    best, bad = -1, 0                               # best val QWK and the patience counter
    save_to = f"{out_dir}/e1_b0_seed{seed}.pt"      # best weights
    log = []                                        # numbers of each epoch

    for ep in range(EPOCHS):
        model.train()                               # training mode
        total = 0                                   # running loss
        for x, lv, y in tqdm(train_loader, desc=f"epoch {ep+1}", leave=False):
            x, lv, y = x.to(DEVICE), lv.to(DEVICE), y.to(DEVICE)   # move to gpu
            opt.zero_grad()                         # clear old gradients
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = coral_loss(model(x).float(), lv, y)         # forward pass and loss
            scaler.scale(loss).backward()           # backward pass
            scaler.step(opt)                        # update weights
            scaler.update()                         # update the scaler
            sched.step()                            # move the learning rate
            total += loss.item()                    # add up the loss
        pv, yv = predict(model, val_loader)         # check on the validation set
        k = qwk(pv, yv)                             # val QWK
        if np.isnan(k):                             # nan would block the save, treat it as 0
            k = 0.0
        log.append((ep + 1, total / len(train_loader), k))         # save the numbers
        print(f"epoch {ep+1:02d} | train loss {total/len(train_loader):.4f} | val QWK {k:.4f}")
        if k > best or ep == 0:                     # first epoch always saves, later only if better
            best, bad = k, 0                        # reset the counter
            torch.save(model.state_dict(), save_to) # keep these weights
        else:
            bad += 1                                # no improvement
            if bad >= PATIENCE:                     # out of patience
                print("early stop")
                break
    print("best val QWK:", round(best, 4))
    return save_to, log                             # path of the best weights and the log

In [8]:
# cell 8 - train seed 123, test on the aptos test split, save all in /kaggle/working/output_123
from sklearn.metrics import recall_score, f1_score, roc_auc_score, balanced_accuracy_score

OUT = f"/kaggle/working/output_{SEED}"              # folder where all outputs go
os.makedirs(OUT, exist_ok=True)                     # make the folder if it is not there

# ---------- training ----------
weights_path, history = train_model(SEED, OUT)      # long part, best weights are saved inside OUT
pd.DataFrame(history, columns=["epoch", "train_loss", "val_qwk"]).to_csv(
    f"{OUT}/e1_log_seed{SEED}.csv", index=False)    # save the epoch log

# ---------- testing ----------
model = CoralEffNet().to(DEVICE)                    # empty model
model.load_state_dict(torch.load(weights_path))     # load the best weights of this seed
pt, yt = predict(model, test_loader)                # threshold probabilities and true grades
pred = (pt > 0.5).sum(1)                            # predicted grades 0-4

cp = np.zeros((len(pt), 5))                         # class probabilities from the thresholds
cp[:, 0] = 1 - pt[:, 0]                             # P(y=0)
for k in range(1, 4):
    cp[:, k] = pt[:, k - 1] - pt[:, k]              # P(y=k)
cp[:, 4] = pt[:, 3]                                 # P(y=4)
cp = np.clip(cp, 1e-6, None)                        # remove tiny negatives
cp = cp / cp.sum(1, keepdims=True)                  # rows sum to 1

# ---------- numbers ----------
acc = (pred == yt).mean()                           # accuracy
q = cohen_kappa_score(yt, pred, weights="quadratic")            # QWK
bal = balanced_accuracy_score(yt, pred)             # balanced accuracy
f1 = f1_score(yt, pred, average="macro")            # macro F1
auc = roc_auc_score(yt, cp, multi_class="ovr", labels=[0, 1, 2, 3, 4])   # macro AUC
sens = recall_score(yt, pred, average=None, labels=[0, 1, 2, 3, 4])      # per-class sensitivity
ref_true, ref_pred = yt >= 2, pred >= 2             # referable DR = grade 2 or more
ref_sens = (ref_pred & ref_true).sum() / ref_true.sum()          # referable sensitivity
ref_spec = (~ref_pred & ~ref_true).sum() / (~ref_true).sum()     # referable specificity

print(f"seed {SEED}")
print("accuracy        :", round(acc, 4))
print("QWK             :", round(q, 4))
print("balanced acc    :", round(bal, 4))
print("macro F1        :", round(f1, 4))
print("macro AUC (ovr) :", round(auc, 4))
print("per-class sens  :", dict(zip(range(5), sens.round(3))))
print("referable sens/spec:", round(ref_sens, 3), round(ref_spec, 3))

# ---------- save predictions and a one-row summary ----------
out = m_test[["id_code", "label", "laterality", "group_id"]].copy()   # keep ids for the CIs later
out["pred"] = pred
for k in range(5):
    out[f"p{k}"] = cp[:, k]
out.to_csv(f"{OUT}/e1_test_preds_seed{SEED}.csv", index=False)        # test predictions

pd.DataFrame([dict(seed=SEED, acc=acc, qwk=q, bal_acc=bal, macro_f1=f1, macro_auc=auc,
                   ref_sens=ref_sens, ref_spec=ref_spec,
                   **{f"sens_{i}": s for i, s in enumerate(sens)})]).to_csv(
    f"{OUT}/e1_summary_seed{SEED}.csv", index=False)                  # one-row summary

# ---------- zip the folder so it is easy to download ----------
shutil.make_archive(OUT, "zip", OUT)                # makes output_123.zip next to the folder
print(os.listdir(OUT))                              # check what was saved
torch.cuda.empty_cache()                            # free gpu memory

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 217MB/s]


epoch 01 | train loss 2.1508 | val QWK 0.5339


epoch 02 | train loss 1.6342 | val QWK 0.8442


epoch 03 | train loss 1.3370 | val QWK 0.8536


...

epoch 08 | train loss 1.1166 | val QWK 0.8870


epoch 09 | train loss 1.0918 | val QWK 0.8697


epoch 10 | train loss 1.0662 | val QWK 0.8726


epoch 11 | train loss 1.0412 | val QWK 0.8802


epoch 12 | train loss 0.9862 | val QWK 0.8619
early stop
best val QWK: 0.8997
seed 2024
accuracy        : 0.7377
QWK             : 0.9063
balanced acc    : 0.5983
macro F1        : 0.5552
macro AUC (ovr) : 0.9394
per-class sens  : {0: np.float64(0.98), 1: np.float64(0.4), 2: np.float64(0.391), 3: np.float64(0.706), 4: np.float64(0.515)}
referable sens/spec: 0.905 0.939
['e1_test_preds_seed2024.csv', 'e1_b0_seed2024.pt', 'e1_summary_seed2024.csv', 'e1_log_seed2024.csv']
